In [4]:
import html
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from matplotlib import style
from wordcloud import WordCloud, STOPWORDS
from sklearn.model_selection import train_test_split

style.use('ggplot')

SEED = 42

# Chỉ dùng khi vẽ WordCloud/thống kê từ,
# không xóa khỏi dữ liệu đưa vào model.
eda_stop_words = set(STOPWORDS) - {
    'not', 'no', 'nor', 'never'
}


In [5]:

#Xử lý các review
def data_processing(text):
    text = html.unescape(str(text))
    text = text.lower()

    # Xóa các biến thể của <br>
    text = re.sub(r'<br\s*/?>', '', text)

    # Xóa HTML tag còn lại
    text = re.sub(r'<[^>]+>', '', text)

    # Xóa URL
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ', text
    )

    # Giữ chữ, số, khoảng trắng và dấu nháy đơn
    text = re.sub( r"[^a-z0-9\s']",' ',text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    return text




```
"This movie is really good"
        ↓
Tokenizer của DistilBERT
        ↓
input_ids + attention_mask
        ↓
DistilBERT
        ↓
Classification Head
        ↓
0 hoặc 1
        ↓
Negative / Positive
```

## 1. Import thư viện

In [6]:
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support)

## 2. Check data

In [7]:
import pandas as pd

train_df = pd.read_csv("train.csv")
validation_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

train_df["clean_review"] = (
    train_df["clean_review"].astype(str)
)
validation_df["clean_review"] = (
    validation_df["clean_review"].astype(str)
)
test_df["clean_review"] = (
    test_df["clean_review"].astype(str)
)

train_df["label"] = train_df["label"].astype("int64")
validation_df["label"] = (
    validation_df["label"].astype("int64")
)
test_df["label"] = test_df["label"].astype("int64")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (34703, 2)
Validation: (7437, 2)
Test: (7437, 2)


In [8]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain:")
display(train_df.head())

print("\nLabel distribution:")
print(train_df['label'].value_counts())

Train shape: (34703, 2)
Validation shape: (7437, 2)
Test shape: (7437, 2)

Train:


,clean_review,label
0,due to the invention of a the domestication co...,1
1,wow so much fun probably a bit much for normal...,1
2,the abc gears up it's repertory company for an...,0
3,being in the suburbs of new york when the z bo...,1
4,this was soul provoking i am an iranian and li...,1



Label distribution:
label
1    17416
0    17287
Name: count, dtype: int64


## 3. Convert từ pandas sang huggingface

### Lý do chuyển đổi
- Tối ưu bộ nhớ
- Tương thích với PyTorch / TensorFlow
<small>_Cấu trúc Dataset của Hugging Face được thiết kế riêng để tự động chuyển đổi dữ liệu thành các Tensor (torch.Tensor hoặc tf.Tensor)_</small>


In [9]:
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)

validation_dataset = Dataset.from_pandas(validation_df, preserve_index=False)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['clean_review', 'label'],
    num_rows: 34703
})
Dataset({
    features: ['clean_review', 'label'],
    num_rows: 7437
})
Dataset({
    features: ['clean_review', 'label'],
    num_rows: 7437
})


## 4. Chọn model

In [10]:
MODEL_NAME = "distilbert-base-uncased"

# 5. Tạo tokenizer

In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# 6. Test tokenizer

In [12]:
example = train_df['clean_review'].iloc[0]

print("Original:")
print(example)

print("\nTokens:")
print(tokenizer.tokenize(example)[:50])

print("\nEncoded:")
print(tokenizer(example))

Original:
due to the invention of a the domestication collar flesh eating zombies are brought under control and become productive members of society however they perform menial tasks the servile dead attend to those living in fenced us 1950s styled small towns while untamed zombies roam around in the wild zone in the town of willard pre teen k'sun ray as timmy robinson lives with parents carrie anne moss and dylan baker as helen and bill robinson alas the robinsons are the only family on their street who do not own a zombie their new neighbors the bottoms have six so to keep up the robinsons obtain zombie billy connolly as fido unfortunately mr connolly's domestication collar is damaged by an old lady's walker and he eats her then new and hungry zombies infest the town meanwhile young ray has grown attached to connolly the boy and his zombie are like tv's timmy and lassie and the robinson family find it difficult to cooperate with the controlling zomcom authorities fido doesn't go far 

## 7. Tokenizer cho dataset

In [13]:
MAX_LENGTH = 128

def tokenize_function(batch):
    return tokenizer(
        batch['clean_review'],
        truncation=True, # Cắt câu dài hơn MAX_LENGTH
        max_length=MAX_LENGTH)

# Gắn số (tra dictionary) cho dataset
tokenized_train = train_dataset.map(tokenize_function, batched=True)

tokenized_validation = validation_dataset.map(tokenize_function, batched=True)

tokenized_test = test_dataset.map(tokenize_function, batched=True)


print(tokenized_train[0])

Map:   0%|          | 0/34703 [00:00<?, ? examples/s]

Map:   0%|          | 0/7437 [00:00<?, ? examples/s]

Map:   0%|          | 0/7437 [00:00<?, ? examples/s]

{'clean_review': "due to the invention of a the domestication collar flesh eating zombies are brought under control and become productive members of society however they perform menial tasks the servile dead attend to those living in fenced us 1950s styled small towns while untamed zombies roam around in the wild zone in the town of willard pre teen k'sun ray as timmy robinson lives with parents carrie anne moss and dylan baker as helen and bill robinson alas the robinsons are the only family on their street who do not own a zombie their new neighbors the bottoms have six so to keep up the robinsons obtain zombie billy connolly as fido unfortunately mr connolly's domestication collar is damaged by an old lady's walker and he eats her then new and hungry zombies infest the town meanwhile young ray has grown attached to connolly the boy and his zombie are like tv's timmy and lassie and the robinson family find it difficult to cooperate with the controlling zomcom authorities fido doesn't

## 8. Padding trong cùng batch

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 9. Load DistilBERT

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2,
    id2label={ # Số sang chữ
        0: "NEGATIVE",
        1: "POSITIVE"
    },
    label2id={ # Chữ sang số
        "NEGATIVE": 0,
        "POSITIVE": 1
    }
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 10. Metric

In [22]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(
        predictions, # Số bên nào lớn hơn thì mô hình nghiêng về nhãn đó
        axis=1 # Tìm vị trí của số lớn nhất trong cặp số đó để chốt câu trả lời cuối cùng của mô hình là 0 hay 1
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary', # Phân loại nhị phân
        zero_division=0 # Ép chia cho 0 tránh crash
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )



    return {
        'accuracy': accuracy, # Độ chính xác tổng thể
        'precision': precision, # Độ chính xác khi đoán nhãn POSITIVE (Thư viện mặc định coi nhãn 1 POSITIVE - mục tiêu (Positive Class), còn nhãn 0 là nhãn phụ)
        'recall': recall, # Nhận diện sót
        'f1': f1, # Điểm số cân bằng giữa precision và recall

    }

## 11. Cấu hình Fine-tuning

In [19]:
training_args = TrainingArguments(
    output_dir="./distilbert_imdb",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=2,
    weight_decay=0.01,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,
    seed=SEED,
    report_to="none")

## 12. Train

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics)

train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.193764,0.499250,0.877235,0.867188,0.892044,0.879440
2,0.118776,0.551984,0.887992,0.881981,0.896866,0.889361


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 13. Validation

In [24]:
validation_results = trainer.evaluate(tokenized_validation)

print(validation_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.118776,0.551984,2,0.887992,0.881981,0.896866,0.889361


{'eval_loss': 0.5519840121269226, 'eval_accuracy': 0.8879924700820223, 'eval_precision': 0.8819810326659642, 'eval_recall': 0.8968657915885347, 'eval_f1': 0.8893611369371762}


## 14. Test

In [25]:
test_results = trainer.evaluate(tokenized_test)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.118776,0.519846,2,0.892564,0.891380,0.894962,0.893168


{'eval_loss': 0.5198456048965454, 'eval_accuracy': 0.8925642059970418, 'eval_precision': 0.8913797704830531, 'eval_recall': 0.894962486602358, 'eval_f1': 0.8931675357668137}


## 15. Bonus: Xử lý kết quả dự đoán

```python
probabilities = torch.softmax(outputs.logits, dim=1)
predicted_label = probabilities.argmax(dim=1).item()
confidence = probabilities.max().item()
```

### `probabilities`

Chuyển điểm thô `logits` của model thành xác suất bằng hàm `softmax`.

```text
Logits:       [-1.2, 2.8]
                  ↓ softmax
Probabilities: [0.018, 0.982]
```

Kết quả:

```text
NEGATIVE: 1.8%
POSITIVE: 98.2%
```

### `predicted_label`

Lấy vị trí của xác suất lớn nhất để xác định nhãn dự đoán.

```text
Probabilities: [0.018, 0.982]
                         ↑ lớn nhất

predicted_label = 1
```

Quy ước:

```text
0 = NEGATIVE
1 = POSITIVE
```

=> `POSITIVE`.

### `confidence`

Lấy xác suất lớn nhất để biểu thị độ tin cậy của model.

```text
max(0.018, 0.982) = 0.982
```

Khi đổi sang phần trăm:

```text
Confidence = 98.2%
```

Tóm tắt:

```text
softmax → chuyển logits thành xác suất
argmax  → chọn nhãn dự đoán
max     → lấy độ tin cậy
```


In [26]:
test_reviews = [
    "This movie was amazing and I really enjoyed it.",
    "This movie was boring and completely terrible."
]

# train() huấn luyện
model.eval() # Chế độ dự đoán

for review in test_reviews:
    cleaned_review = data_processing(review)

    inputs = tokenizer( # Tokenize văn bản
        cleaned_review,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)
    predicted_label = probabilities.argmax(dim=1).item()
    confidence = probabilities.max().item()

    sentiment = "POSITIVE" if predicted_label == 1 else "NEGATIVE"

    print(f"Review: {review}")
    print(f"Prediction: {sentiment} ({confidence:.2%})")
    print()

Review: This movie was amazing and I really enjoyed it.
Prediction: POSITIVE (99.95%)

Review: This movie was boring and completely terrible.
Prediction: NEGATIVE (99.96%)

